In [1]:
import os

from aicsimageio import AICSImage

# Loading image into lib

In [2]:
run = "run_1"

PATHS = {
    "run_1": "/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/Xenium_PDO_run_2_1_HE_16-07-2025.czi", 
    "run_2": "/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/Xenium_PDO_run_2_2_HE-16-07-2025.czi"
}

CORRESPONDENCES = {
    "run_1": [
        "1HVQ",
        "1CNN", 
        "077I", 
        "1GAA",
        "1J25",
        "131N",
        "OWJ3",
        "14PT",
    ],
    "run_2": [
        "169V",
        "1BI7",
        "1CI5",
        "1FMS",
        "12NM",
        "OLR9",
        "1GVB",
        "1GNS"
    ]
}

img = AICSImage(PATHS[run])
img.scenes

('ScanRegion0',
 'ScanRegion1',
 'ScanRegion2',
 'ScanRegion3',
 'ScanRegion4',
 'ScanRegion5',
 'ScanRegion6',
 'ScanRegion7')

In [5]:
os.makedirs(f"/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/ome_tiff/{run}", exist_ok=True)
for index, patient_id in enumerate(CORRESPONDENCES[run]):
    image_id = img.scenes[index]
    img.set_scene(image_id)
    img_data = img.get_image_dask_data()
    print(f"Image {image_id} has shape {img_data.shape}")
    # ome_tiff_path = f"/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/ome_tiff/{run}/{image_id}.ome.tiff"
    # img.save(ome_tiff_path)
    # print(f"Saved OME-TIFF to {ome_tiff_path}")

    break

Image ScanRegion0 has shape (1, 1, 1, 64020, 53449, 3)


In [6]:
img.dims

<Dimensions [T: 1, C: 1, Z: 1, Y: 64020, X: 53449, S: 3]>

In [3]:
import os
import numpy as np
from czifile import CziFile
import tifffile

def process_image(run, index):
    """Process a specific image from a run based on index using czifile"""
    if run not in PATHS:
        raise ValueError(f"Invalid run: {run}. Available runs: {list(PATHS.keys())}")
    
    if index >= len(CORRESPONDENCES[run]):
        raise ValueError(f"Index {index} out of range for run {run}. Max index: {len(CORRESPONDENCES[run]) - 1}")
    
    # Get the patient ID for the given index
    patient_id = CORRESPONDENCES[run][index]
    
    # Load the CZI file
    with CziFile(PATHS[run]) as czi:
        import pdb; pdb.set_trace()
        # Get all sc
        # enes (subblocks)
        # scenes = czi.subblock_directory
        # print("scenes", scenes)
        # print("scenes", [scene.__dir__() for scene in scenes])
        
        if index >= len(scenes):
            raise ValueError(f"Index {index} out of range for CZI file. File contains {len(scenes)} scenes.")
        
        # Get the specific scene (subblock) data
        scene_data = czi.subblock_directory[index]
        print(f"Processing scene {index} with data {scene_data} for Patient ID: {patient_id}")

        image_data = czi.imread(subblock=index)
        
        # Remove singleton dimensions and get image data in TCZYX order
        # CZI data typically has dimensions: [T, C, Z, Y, X] or similar
        image_data = np.squeeze(image_data)
        
        print(f"Image for Patient ID: {patient_id} has shape {image_data.shape} and dtype {image_data.dtype}")
        
        # Create output directory
        os.makedirs(f"/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/ome_tiff/{run}", exist_ok=True)
        
        # Save as OME-TIFF
        ome_tiff_path = f"/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/ome_tiff/{run}/{patient_id}.ome.tiff"
        
        # Save with OME-TIFF metadata
        tifffile.imwrite(
            ome_tiff_path,
            image_data,
            ome=True,
            metadata={
                'Pixels': {
                    'PhysicalSizeX': scene_data.x_size if hasattr(scene_data, 'x_size') else None,
                    'PhysicalSizeY': scene_data.y_size if hasattr(scene_data, 'y_size') else None,
                    'PhysicalSizeZ': scene_data.z_size if hasattr(scene_data, 'z_size') else None,
                }
            }
        )
        
        print(f"Saved OME-TIFF to {ome_tiff_path}")
    
    return ome_tiff_path

im = process_image("run_1", 0)

> /tmp/ipykernel_3720741/551611495.py(25)process_image()
     23         # print("scenes", [scene.__dir__() for scene in scenes])
     24 
---> 25         if index >= len(scenes):
     26             raise ValueError(f"Index {index} out of range for CZI file. File contains {len(scenes)} scenes.")
     27 

(8, 1, 141635, 251396, 3)
*** AttributeError: 'CziFile' object has no attribute 'storage_size'
*** NameError: name 'czifile' is not defined
*** NameError: name 'czifile' is not defined
'<ImageDocument>\n  <Metadata>\n    <Experiment Version="1.2">\n      <RunMode>ValidateAndAdaptBeforePerformEnabled,OptimizeBeforePerformEnabled,PreventMissingCalibrationDataInformation,PreventStageXYRestoreAfterExperiment,PreventFocusRestoreAfterExperiment</RunMode>\n      <BeforeHardwareSetting>HardwareBefore</BeforeHardwareSetting>\n      <AfterHardwareSetting>HardwareAfter</AfterHardwareSetting>\n      <ExperimentBlockIndex>0</ExperimentBlockIndex>\n      <IsSegmented>false</IsSegmented>\n      <Is

BdbQuit: 

In [4]:
import os
import argparse
from aicsimageio import AICSImage
from aicsimageio.writers import OmeTiffWriter


In [ ]:
def process_image(run, index):
    """Process a specific image from a run based on index"""
    if run not in PATHS:
        raise ValueError(f"Invalid run: {run}. Available runs: {list(PATHS.keys())}")
    
    if index >= len(CORRESPONDENCES[run]):
        raise ValueError(f"Index {index} out of range for run {run}. Max index: {len(CORRESPONDENCES[run]) - 1}")
    
    # Load the image
    img = AICSImage(PATHS[run])
    
    # Get the scene ID for the given index
    scene_id = img.scenes[index]
    patient_id = CORRESPONDENCES[run][index]
    
    # Set the scene and get image data
    img.set_scene(scene_id)
    print(img.shape)
    img_data = img.get_image_data("TCZYX")
    import pdb; pdb.set_trace()
    img_data_shape = img_data.shape[-2:]
    print(f"Image {scene_id} (Patient ID: {patient_id}) has shape {img_data.shape}")
    
    # Create output directory
    os.makedirs(f"/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/ome_tiff/{run}", exist_ok=True)
    
    # Save as OME-TIFF
    ome_tiff_path = f"/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/ome_tiff/{run}/{patient_id}.ome.tiff"
    img.save(ome_tiff_path, select_scenes=[scene_id])
    print(f"Saved OME-TIFF to {ome_tiff_path}")

    im = tifffile.imread(ome_tiff_path)
    print(im.shape, img_data_shape)

    return ome_tiff_path

In [22]:
process_image("run_1", 6)

(1, 1, 1, 50806, 47924, 3)
> /tmp/ipykernel_3720741/1537761425.py(21)process_image()
     19     img_data = img.get_image_data("TCZYX")
     20     import pdb; pdb.set_trace()
---> 21     img_data_shape = img_data.shape[-2:]
     22     print(f"Image {scene_id} (Patient ID: {patient_id}) has shape {img_data.shape}")
     23 



BdbQuit: 